In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
data = np.load("spectrum_sensing_features.npz")

X = data["X"]
y = data["y"]
snr = data["snr"]

indices = np.arange(len(y))

stratify_groups = np.array([
    f"{label}_{s}"
    for label, s in zip(y, snr)
])

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=42,
    stratify=stratify_groups
)

temp_groups = np.array([
    f"{y[i]}_{snr[i]}"
    for i in temp_idx
])

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    stratify=temp_groups
)

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

snr_train = snr[train_idx]
snr_val = snr[val_idx]
snr_test = snr[test_idx]

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (9800, 13)
Validation: (2100, 13)
Testing: (2100, 13)


In [3]:
print("Training class distribution:")
print(pd.Series(y_train).value_counts())

print("\nValidation class distribution:")
print(pd.Series(y_val).value_counts())

print("\nTesting class distribution:")
print(pd.Series(y_test).value_counts())

Training class distribution:
1    4900
0    4900
Name: count, dtype: int64

Validation class distribution:
0    1050
1    1050
Name: count, dtype: int64

Testing class distribution:
1    1050
0    1050
Name: count, dtype: int64


In [4]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

print("Random Forest model initialised successfully!")

Random Forest model initialised successfully!


In [5]:
rf_model.fit(X_train, y_train)

print("Random Forest training completed!")

Random Forest training completed!


Random Forest model and threshold saved.
Threshold: 0.62


In [6]:
# Predict class labels
y_val_pred = rf_model.predict(X_val)

# Predict probability of H1
y_val_prob = rf_model.predict_proba(X_val)[:, 1]

# Calculate preliminary accuracy
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Preliminary Validation Accuracy: {val_accuracy:.4f}")
print("Prediction shape:", y_val_pred.shape)
print("Probability shape:", y_val_prob.shape)

Preliminary Validation Accuracy: 0.8338
Prediction shape: (2100,)
Probability shape: (2100,)


In [7]:
target_pfa = 0.05

# Select H0 validation probabilities
h0_val_prob = y_val_prob[y_val == 0]

# Calculate the 95th percentile threshold
rf_threshold = np.quantile(
    h0_val_prob,
    1 - target_pfa,
    method="higher"
)

print("Random Forest threshold:", rf_threshold)

Random Forest threshold: 0.62


In [8]:
y_val_calibrated = (
    y_val_prob > rf_threshold
).astype(int)

val_h0_mask = y_val == 0

val_pfa = np.mean(
    y_val_calibrated[val_h0_mask] == 1
)

print(f"Target Pfa: {target_pfa:.4f}")
print(f"Actual Validation Pfa: {val_pfa:.4f}")

Target Pfa: 0.0500
Actual Validation Pfa: 0.0467


In [9]:
val_h1_mask = y_val == 1

val_pd = np.mean(
    y_val_calibrated[val_h1_mask] == 1
)

print(f"Validation Pd: {val_pd:.4f}")
print(f"Validation Pm: {1 - val_pd:.4f}")

Validation Pd: 0.7019
Validation Pm: 0.2981


In [10]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_auc_score
)

# Predict probabilities on test data
y_test_prob = rf_model.predict_proba(X_test)[:, 1]

# Apply validation-calibrated threshold
y_test_pred = (y_test_prob > rf_threshold).astype(int)

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(
    y_test, y_test_pred, labels=[0, 1]
).ravel()

# Calculate metrics
test_accuracy = accuracy_score(y_test, y_test_pred)
test_pd = tp / (tp + fn)
test_pfa = fp / (fp + tn)
test_pm = fn / (fn + tp)
rf_auc = roc_auc_score(y_test, y_test_prob)

print("RANDOM FOREST - TEST RESULTS")
print("----------------------------")
print(f"Threshold: {rf_threshold:.4f}")
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Pd:        {test_pd:.4f}")
print(f"Pfa:       {test_pfa:.4f}")
print(f"Pm:        {test_pm:.4f}")
print(f"ROC-AUC:   {rf_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

RANDOM FOREST - TEST RESULTS
----------------------------
Threshold: 0.6200
Accuracy:  0.8219
Pd:        0.6990
Pfa:       0.0552
Pm:        0.3010
ROC-AUC:   0.9383

Confusion Matrix:
[[992  58]
 [316 734]]


In [11]:
rf_results = []

for s in np.unique(snr_test):
    mask = snr_test == s

    tn, fp, fn, tp = confusion_matrix(
        y_test[mask],
        y_test_pred[mask],
        labels=[0, 1]
    ).ravel()

    pd_value = tp / (tp + fn)
    pfa_value = fp / (fp + tn)
    pm_value = fn / (fn + tp)

    rf_results.append({
        "SNR": s,
        "Pd": pd_value,
        "Pfa": pfa_value,
        "Pm": pm_value
    })

rf_df = pd.DataFrame(rf_results)

print(rf_df.round(4))

   SNR      Pd     Pfa      Pm
0  -20  0.0867  0.0933  0.9133
1  -15  0.2200  0.0867  0.7800
2  -10  0.6200  0.2000  0.3800
3   -5  0.9933  0.0067  0.0067
4    0  1.0000  0.0000  0.0000
5    5  1.0000  0.0000  0.0000
6   10  0.9733  0.0000  0.0267


In [12]:
rf_df.to_csv("random_forest_results.csv", index=False)

In [13]:
rf_summary = pd.DataFrame([{
    "Model": "Random Forest",
    "Threshold": rf_threshold,
    "Accuracy": test_accuracy,
    "Pd": test_pd,
    "Pfa": test_pfa,
    "Pm": test_pm,
    "ROC_AUC": rf_auc
}])

rf_summary.to_csv("random_forest_summary.csv", index=False)

print(rf_summary.round(4))

           Model  Threshold  Accuracy     Pd     Pfa     Pm  ROC_AUC
0  Random Forest       0.62    0.8219  0.699  0.0552  0.301   0.9383


In [15]:
import joblib
import numpy as np

# Save original trained model
joblib.dump(rf_model, "random_forest_model.pkl")

# Save original calibrated threshold
np.save("random_forest_threshold.npy", rf_threshold)

print("Random Forest model and threshold saved.")
print("Threshold:", rf_threshold)

Random Forest model and threshold saved.
Threshold: 0.62
